In [3]:
#library(IRkernel)
#IRkernel::installspec(name = 'python_r_env', displayname = 'R (python_r_env)')
#Sys.which("R")

In [4]:
#================================================================================
# Load Libraries and Environment Variables
#================================================================================
library(dplyr)
library(here)
library(poLCA)
library(dotenv)
library(readxl)
library(writexl)

# Load environment variables from the .env file
env_file <- normalizePath("../.env", mustWork = FALSE) # Ensure relative path works
if (!file.exists(env_file)) {
  stop(paste("Environment file not found at:", env_file))
}
load_dot_env(env_file)

# Define workspace path
WORKSPACE_PATH <- Sys.getenv("WORKSPACE_PATH")
WORKSPACE_PATH <- normalizePath(WORKSPACE_PATH)

# Source utility functions
source(file.path(WORKSPACE_PATH, "src/config/utils.R"))

DATA_DIR <- file.path(WORKSPACE_PATH, "data")
MOMENT_OF_SUICIDE_FEATURES <- split_string(Sys.getenv("MOMENT_OF_SUICIDE_FEATURES"))
SOCIO_DEMOGRAPHIC_FEATURES <- split_string(Sys.getenv("SOCIO_DEMOGRAPHIC_FEATURES"))



Dołączanie pakietu: 'openxlsx2'


Następujący obiekt został zakryty z 'package:writexl':

    write_xlsx


Następujący obiekt został zakryty z 'package:readxl':

    read_xlsx




In [6]:
#================================================================================
# Load Data
#================================================================================

# File paths
encoded_data_path <- file.path(WORKSPACE_PATH, "data/processed/encoded_data.csv")
group_set_path <- file.path(WORKSPACE_PATH, "data/processed/group_set.csv")

# Read data
encoded_data <- read_csv(encoded_data_path)
group_set <- read_csv(group_set_path)

# Combine feature prefixes
prefixes_to_lca <- unique(c(MOMENT_OF_SUICIDE_FEATURES, SOCIO_DEMOGRAPHIC_FEATURES))

# Select columns matching prefixes
columns_to_lca <- grep(paste0("^(", paste(prefixes_to_lca, collapse = "|"), ")"), 
                        names(encoded_data), value = TRUE)

# Include "ID" column
selected_columns <- c(columns_to_lca, "ID")
encoded_data <- encoded_data[, selected_columns]

# Add missing columns from group_set to encoded_data
new_columns <- setdiff(names(group_set), names(encoded_data))
if (length(new_columns) > 0) {
  encoded_data <- left_join(
    encoded_data,
    group_set %>% dplyr::select(all_of(c("ID", new_columns))),
    by = "ID"
  )
} else {
  message("No new columns to join.")
}

# Modify values: 0 -> 1, 1 -> 2
encoded_data <- encoded_data %>%
  mutate(across(all_of(columns_to_lca), ~ if_else(. == 0, 1, if_else(. == 1, 2, .))))

#================================================================================
# Define LCA Functions
#================================================================================

# Perform LCA for a single subset of data
perform_lca <- function(data, n_classes) {
  # Perform Latent Class Analysis (LCA) and return model statistics.
  #
  # Args:
  #   data: A data frame containing numeric variables for LCA.
  #   n_classes: Number of latent classes to fit.
  #
  # Returns:
  #   A data frame with:
  #     - Predicted_Class: Predicted latent classes for each row.
  #     - AIC: Akaike Information Criterion.
  #     - BIC: Bayesian Information Criterion.
  #     - G_squared: Likelihood ratio/deviance statistic (G²).
  #     - X_squared: Chi-square goodness of fit statistic (X²).

  # Validate inputs
  if (!is.data.frame(data) || !all(sapply(data, is.numeric))) {
    stop("Data for LCA must be a numeric data frame.")
  }

  # Create LCA formula
  lca_formula <- as.formula(paste("cbind(", paste(names(data), collapse = ", "), ") ~ 1"))

  # Perform LCA
  lca_result <- poLCA::poLCA(lca_formula, data = data, nclass = n_classes, na.rm = TRUE)

  # Extract results
  data.frame(
    Predicted_Class = lca_result$predclass,
    AIC = lca_result$aic,
    BIC = lca_result$bic,
    Gsq = lca_result$Gsq,
    Chisq = lca_result$Chisq
  )
}


run_latent_class_analysis <- function(data, group_column, n_classes = 5, columns_to_lca = NULL) {
  # Perform latent class analysis (LCA) for grouped data.
  #
  # Args:
  #   data: A data frame with grouped data.
  #   group_column: The column to group by for LCA.
  #   n_classes: Number of latent classes to fit.
  #   columns_to_lca: A character vector of column names to use for LCA.
  # Returns:
  #   A data frame with "ID", LCA class, AIC, and BIC.

  # Validate inputs
  if (!"ID" %in% names(data)) stop("Data must contain an 'ID' column.")
  if (!group_column %in% names(data)) stop(paste("Column", group_column, "not found in data."))

  # If columns_to_lca is not specified, use all columns except "ID" and group_column
  if (is.null(columns_to_lca)) {
    columns_to_lca <- setdiff(names(data), c("ID", group_column))
  }

  # Validate columns_to_lca
  if (!all(columns_to_lca %in% names(data))) {
    invalid_cols <- columns_to_lca[!columns_to_lca %in% names(data)]
    stop("The following columns in 'columns_to_lca' do not exist in the data: ", paste(invalid_cols, collapse = ", "))
  }

  # Get unique group values
  group_values <- unique(data[[group_column]])
  final_results <- data.frame()

  # Loop through group values and perform LCA
  for (group_value in group_values) {
    # Filter rows for the current group
    filtered_rows <- data %>% filter(.data[[group_column]] == group_value)

    # Extract ID column and subset columns for LCA
    id_column <- filtered_rows$ID
    filtered_rows <- filtered_rows[, columns_to_lca, drop = FALSE]

    # Perform LCA
    lca_result <- perform_lca(filtered_rows, n_classes)

    # Sort classes by frequency
    class_frequencies <- table(lca_result$Predicted_Class)
    sorted_classes <- as.integer(names(sort(class_frequencies, decreasing = TRUE)))
    remapped_classes <- match(lca_result$Predicted_Class, sorted_classes)

    # Create a temporary data frame for the group
    temp_results <- data.frame(
      ID = id_column,
      Predicted_Class = remapped_classes,
      AIC = lca_result$AIC[1],
      BIC = lca_result$BIC[1],
      Gsq = lca_result$Gsq[1],
      Chisq = lca_result$Chisq[1]
    )

    # Append to final results
    final_results <- rbind(final_results, temp_results)
  }

  # Rename columns with the required naming convention
  colnames(final_results) <- c(
    "ID",
    paste0("LCA_", group_column, "_class"),
    paste0("LCA_", group_column, "_AIC"),
    paste0("LCA_", group_column, "_BIC"),
    paste0("LCA_", group_column, "_Gsq"),
    paste0("LCA_", group_column, "_Chisq")
  )

  return(final_results)
}


#================================================================================
# Execute LCA
#================================================================================
group_columns <- c("Group_AF", "Group_AG", "Group_AGF")

for (group_column in group_columns) {
  # Define columns to exclude based on the group column
  excluded_columns <- switch(
    group_column,
    "Group_AF" = "Fatal",
    "Group_AG" = "Gender",
    "Group_AGF" = c("Fatal", "Gender"),
    NULL # Default case if no exclusions are needed
  )

  # Select columns to use in LCA
  selected_columns <- columns_to_lca[!columns_to_lca %in% excluded_columns]

  # Run latent class analysis
  lca_classes <- run_latent_class_analysis(
    data = encoded_data,
    group_column = group_column,
    n_classes = 5,
    columns_to_lca = selected_columns
  )

  # Save results
  output_path <- file.path(WORKSPACE_PATH, "data/processed/lca_group_results.xlsx")
  write_excel(output_path, lca_classes, sheet_name = group_column, if_sheet_exists = "replace")
}


Conditional item response (column) probabilities,
 by outcome variable, for each class (row) 
 
$Gender
           Pr(1)  Pr(2)
class 1:  0.0602 0.9398
class 2:  0.0620 0.9380
class 3:  0.6428 0.3572
class 4:  0.3777 0.6223
class 5:  0.4336 0.5664

$Marital_Cohabitant
           Pr(1)  Pr(2)
class 1:  1.0000 0.0000
class 2:  0.8708 0.1292
class 3:  0.9789 0.0211
class 4:  0.9401 0.0599
class 5:  0.9299 0.0701

$Marital_Cohabiting
           Pr(1)  Pr(2)
class 1:  1.0000 0.0000
class 2:  0.9901 0.0099
class 3:  0.9951 0.0049
class 4:  0.9900 0.0100
class 5:  0.9899 0.0101

$Marital_Divorced
           Pr(1)  Pr(2)
class 1:  1.0000 0.0000
class 2:  0.9542 0.0458
class 3:  0.9978 0.0022
class 4:  0.9713 0.0287
class 5:  0.9679 0.0321

$Marital_Married
           Pr(1)  Pr(2)
class 1:  1.0000 0.0000
class 2:  0.6883 0.3117
class 3:  0.9785 0.0215
class 4:  0.8263 0.1737
class 5:  0.8522 0.1478

$Marital_Separated
           Pr(1)  Pr(2)
class 1:  1.0000 0.0000
class 2:  0.9959 0.0041
class

Data successfully written to: C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study/data/processed/lca_group_results.xlsx in sheet: Group_AF



Conditional item response (column) probabilities,
 by outcome variable, for each class (row) 
 
$Marital_Cohabitant
           Pr(1)  Pr(2)
class 1:  0.9073 0.0927
class 2:  0.9388 0.0612
class 3:  1.0000 0.0000
class 4:  0.9271 0.0729
class 5:  0.9458 0.0542

$Marital_Cohabiting
           Pr(1)  Pr(2)
class 1:  0.9930 0.0070
class 2:  0.9959 0.0041
class 3:  1.0000 0.0000
class 4:  0.9920 0.0080
class 5:  0.9942 0.0058

$Marital_Divorced
           Pr(1)  Pr(2)
class 1:  0.9679 0.0321
class 2:  0.9677 0.0323
class 3:  1.0000 0.0000
class 4:  0.9696 0.0304
class 5:  0.9763 0.0237

$Marital_Married
           Pr(1)  Pr(2)
class 1:  0.7356 0.2644
class 2:  0.8523 0.1477
class 3:  1.0000 0.0000
class 4:  0.8034 0.1966
class 5:  0.8903 0.1097

$Marital_Separated
           Pr(1)  Pr(2)
class 1:  0.9969 0.0031
class 2:  0.9977 0.0023
class 3:  1.0000 0.0000
class 4:  0.9980 0.0020
class 5:  0.9988 0.0012

$Marital_Single
           Pr(1)  Pr(2)
class 1:  0.3998 0.6002
class 2:  0.2481 0.75

Data successfully written to: C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study/data/processed/lca_group_results.xlsx in sheet: Group_AG



Conditional item response (column) probabilities,
 by outcome variable, for each class (row) 
 
$Marital_Cohabitant
           Pr(1)  Pr(2)
class 1:  0.8392 0.1608
class 2:  0.9215 0.0785
class 3:  1.0000 0.0000
class 4:  0.9966 0.0034
class 5:  0.9252 0.0748

$Marital_Cohabiting
           Pr(1)  Pr(2)
class 1:  0.9850 0.0150
class 2:  0.9936 0.0064
class 3:  1.0000 0.0000
class 4:  1.0000 0.0000
class 5:  0.9913 0.0087

$Marital_Divorced
           Pr(1)  Pr(2)
class 1:  0.9479 0.0521
class 2:  0.9729 0.0271
class 3:  1.0000 0.0000
class 4:  1.0000 0.0000
class 5:  0.9591 0.0409

$Marital_Married
           Pr(1)  Pr(2)
class 1:  0.5880 0.4120
class 2:  0.8910 0.1090
class 3:  1.0000 0.0000
class 4:  0.9978 0.0022
class 5:  0.8212 0.1788

$Marital_Separated
           Pr(1)  Pr(2)
class 1:  0.9957 0.0043
class 2:  0.9985 0.0015
class 3:  1.0000 0.0000
class 4:  1.0000 0.0000
class 5:  0.9985 0.0015

$Marital_Single
           Pr(1)  Pr(2)
class 1:  0.6456 0.3544
class 2:  0.2225 0.77

Data successfully written to: C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study/data/processed/lca_group_results.xlsx in sheet: Group_AGF

